# Mini-project — Leakage-safe Persian learner-corpus preprocessor

## Mission

Turn a small raw learner-corpus table into model-ready train, validation, and test feature
matrices. The project preserves aligned text views, audits representation and quality,
keeps related responses together, and fits every learned transformation on training data.

**Deliverables**

- a documented feature contract;
- raw, normalized, and analysis text views;
- missingness, class-balance, and duplicate audits;
- disjoint group-aware partitions;
- one mixed numerical/categorical/text preprocessor;
- validation checks and a short preprocessing report.

No predictive model is fitted here. Stage 6 will attach estimators to this pipeline.

In [ ]:
import hashlib
import re
import unicodedata

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

rng = np.random.default_rng(1403)

## 1. Create a self-contained teaching corpus

Each learner contributes two responses. Four learners share one collection group, and
repeated prompt/template clusters remain inside that group. Real projects should load a
versioned dataset and document its sampling process.

In [ ]:
topics = [
    "دانشگاه", "کتابخانه", "سفر", "دوستی", "آموزش", "فناوری",
    "خانواده", "ورزش", "موسیقی", "محیط‌زیست", "کار", "شهر",
]
templates = [
    "من درباره {topic} فارسی می‌نویسم",
    "یادگیری واژه‌های {topic} برای من جالب است",
]

rows = []
for learner_number in range(48):
    learner_id = f"L{learner_number + 1:03d}"
    group_id = f"G{learner_number // 4 + 1:02d}"
    topic = topics[learner_number // 4]
    proficiency = rng.choice(["A2", "B1", "B2"], p=[0.28, 0.50, 0.22])
    first_language = rng.choice(["Arabic", "Kurdish", "Turkish"], p=[0.70, 0.20, 0.10])
    gender = rng.choice(["woman", "man"], p=[0.35, 0.65])
    age = float(rng.integers(18, 45))
    if learner_number % 11 == 0:
        age = np.nan

    for response_number, template in enumerate(templates, start=1):
        text = template.format(topic=topic)
        if learner_number % 3 == 0:
            text = text.replace("ی", "ي").replace("ک", "ك")
        if learner_number % 7 == 0 and response_number == 2:
            text += " خیلییی خوبه 😊"

        task_type = rng.choice(["free", "picture", "social"], p=[0.45, 0.35, 0.20])
        response_seconds = float(rng.integers(45, 260))
        review_probability = {"A2": 0.62, "B1": 0.38, "B2": 0.20}[proficiency]
        if task_type == "social":
            review_probability += 0.08

        rows.append({
            "response_id": f"R{len(rows) + 1:03d}",
            "learner_id": learner_id,
            "group_id": group_id,
            "duplicate_cluster": f"{group_id}_T{response_number}",
            "text_raw": text,
            "task_type": task_type,
            "proficiency": proficiency,
            "age": age,
            "response_seconds": response_seconds,
            "first_language": first_language,
            "gender": gender,
            "needs_review": int(rng.random() < review_probability),
        })

corpus = pd.DataFrame(rows)
corpus.head()

## 2. Define conservative Persian text views

In [ ]:
ARABIC_TO_PERSIAN = str.maketrans({"ي": "ی", "ى": "ی", "ك": "ک"})
INVISIBLE_PATTERN = re.compile(r"[\u200e\u200f\u202a-\u202e\u2066-\u2069]")
SPACE_PATTERN = re.compile(r"[ \t\r\f\v]+")
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
MENTION_PATTERN = re.compile(r"(?<!\w)@[\w_]+", flags=re.UNICODE)

def normalize_persian(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text))
    text = text.translate(ARABIC_TO_PERSIAN)
    text = INVISIBLE_PATTERN.sub("", text)
    text = re.sub(r"\s*\u200c\s*", "\u200c", text)
    return SPACE_PATTERN.sub(" ", text).strip()

def make_analysis_view(text: str) -> str:
    text = normalize_persian(text)
    text = URL_PATTERN.sub(" <URL> ", text)
    text = MENTION_PATTERN.sub(" <USER> ", text)
    return SPACE_PATTERN.sub(" ", text).strip()

corpus["text_normalized"] = corpus["text_raw"].map(normalize_persian)
corpus["text_analysis"] = corpus["text_raw"].map(make_analysis_view)
corpus[["text_raw", "text_normalized", "text_analysis"]].head(8)

## 3. Write the feature contract

In [ ]:
target = "needs_review"
numeric_features = ["age", "response_seconds"]
categorical_features = ["task_type", "proficiency"]
text_feature = "text_normalized"
identifiers = ["response_id", "learner_id", "group_id", "duplicate_cluster"]
audit_only = ["first_language", "gender"]

model_features = numeric_features + categorical_features + [text_feature]
assert target not in model_features
assert set(model_features).isdisjoint(identifiers + audit_only)

contract = pd.Series({
    "unit": "one learner response",
    "target": target,
    "model_features": model_features,
    "identifiers_and_split_groups": identifiers,
    "audit_only": audit_only,
})
contract

## 4. Audit before splitting

In [ ]:
corpus["normalized_hash"] = corpus["text_normalized"].map(
    lambda text: hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]
)

quality_report = pd.DataFrame({
    "dtype": corpus.dtypes.astype(str),
    "missing_n": corpus.isna().sum(),
    "missing_pct": corpus.isna().mean().mul(100).round(1),
    "unique_n": corpus.nunique(dropna=False),
})

print("Shape:", corpus.shape)
print("Duplicate response IDs:", corpus["response_id"].duplicated().sum())
print("Repeated normalized texts:", corpus.duplicated("normalized_hash", keep=False).sum())
print("Target distribution:", corpus[target].value_counts(normalize=True).round(3).to_dict())
quality_report

In [ ]:
representation = pd.crosstab(
    [corpus["first_language"], corpus["gender"]],
    corpus["proficiency"],
    margins=True,
)
representation

`first_language` and `gender` are retained for sampling and slice audits but excluded from
the feature matrix. This is a design decision, not a universal rule.

## 5. Create disjoint train, validation, and test groups

In [ ]:
outer = GroupShuffleSplit(n_splits=1, train_size=0.67, random_state=42)
train_idx, temp_idx = next(outer.split(corpus, y=corpus[target], groups=corpus["group_id"]))

train = corpus.iloc[train_idx].reset_index(drop=True)
temp = corpus.iloc[temp_idx].reset_index(drop=True)

inner = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=43)
validation_idx, test_idx = next(inner.split(temp, y=temp[target], groups=temp["group_id"]))
validation = temp.iloc[validation_idx].reset_index(drop=True)
test = temp.iloc[test_idx].reset_index(drop=True)

split_groups = {
    "train": set(train["group_id"]),
    "validation": set(validation["group_id"]),
    "test": set(test["group_id"]),
}
assert split_groups["train"].isdisjoint(split_groups["validation"])
assert split_groups["train"].isdisjoint(split_groups["test"])
assert split_groups["validation"].isdisjoint(split_groups["test"])

print("Rows:", {"train": len(train), "validation": len(validation), "test": len(test)})
print("Groups:", {name: len(groups) for name, groups in split_groups.items()})

In [ ]:
split_summary = pd.DataFrame({
    "rows": [len(train), len(validation), len(test)],
    "groups": [train["group_id"].nunique(), validation["group_id"].nunique(), test["group_id"].nunique()],
    "review_rate": [train[target].mean(), validation[target].mean(), test[target].mean()],
    "missing_age_pct": [train["age"].isna().mean() * 100, validation["age"].isna().mean() * 100, test["age"].isna().mean() * 100],
}, index=["train", "validation", "test"]).round(2)
split_summary

## 6. Build the mixed-feature preprocessor

In [ ]:
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])

text_pipeline = FeatureUnion([
    ("word", TfidfVectorizer(
        preprocessor=normalize_persian,
        token_pattern=r"(?u)\b\w+\b",
        ngram_range=(1, 2),
        min_df=2,
        max_features=250,
        sublinear_tf=True,
    )),
    ("character", TfidfVectorizer(
        preprocessor=normalize_persian,
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_features=250,
        sublinear_tf=True,
    )),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
    ("text", text_pipeline, text_feature),
])

## 7. Fit on training rows and transform the held-out rows

In [ ]:
X_train_ready = preprocessor.fit_transform(train[model_features])
X_validation_ready = preprocessor.transform(validation[model_features])
X_test_ready = preprocessor.transform(test[model_features])

y_train = train[target].to_numpy()
y_validation = validation[target].to_numpy()
y_test = test[target].to_numpy()

assert X_train_ready.shape[1] == X_validation_ready.shape[1] == X_test_ready.shape[1]
assert len(y_train) == X_train_ready.shape[0]
assert len(y_validation) == X_validation_ready.shape[0]
assert len(y_test) == X_test_ready.shape[0]

print("Feature matrix shapes:")
print(" train:", X_train_ready.shape)
print(" validation:", X_validation_ready.shape)
print(" test:", X_test_ready.shape)

In [ ]:
feature_names = preprocessor.get_feature_names_out()

def matrix_density(matrix) -> float:
    nonzero = matrix.nnz if hasattr(matrix, "nnz") else np.count_nonzero(matrix)
    return nonzero / (matrix.shape[0] * matrix.shape[1])

print("Number of features:", len(feature_names))
print("Training density:", round(matrix_density(X_train_ready), 4))
print("First feature names:", feature_names[:15].tolist())

## 8. Validate the preprocessing contract

In [ ]:
def assert_no_group_overlap(*frames: pd.DataFrame) -> None:
    group_sets = [set(frame["group_id"]) for frame in frames]
    for left in range(len(group_sets)):
        for right in range(left + 1, len(group_sets)):
            assert group_sets[left].isdisjoint(group_sets[right])

assert_no_group_overlap(train, validation, test)
assert not np.isnan(X_train_ready.data if hasattr(X_train_ready, "data") else X_train_ready).any()
assert target not in model_features
assert set(audit_only).isdisjoint(model_features)

print("All preprocessing checks passed.")

## 9. Preprocessing report

- **Unit:** one learner response.
- **Target:** `needs_review`.
- **Split:** collection groups keep related learners and repeated templates together.
- **Numerical features:** median imputation, missingness indicators, standardization.
- **Categorical features:** explicit missing category and unknown-safe one-hot encoding.
- **Text:** preserved raw view; conservative Unicode/spacing normalization; word and character TF–IDF fitted on training text.
- **Excluded from features:** identifiers, split groups, first language, and gender.
- **Held-out policy:** validation supports development; test remains locked for final Stage 6 evaluation.

## Extensions

1. Replace synthetic rows with a versioned learner-corpus file.
2. Build connected split groups when learner and near-duplicate clusters cross one another.
3. Add a documented Persian tokenizer and compare it with character n-grams.
4. Add schema checks for allowed labels, ranges, and required columns.
5. In Stage 6, attach a baseline classifier inside the same pipeline and use group-aware cross-validation.